# 13. Debugging · Quality · Config

문제를 빠르게 찾고, 문서 품질을 자동 점검하고, App 기본 설정을 공유하는 도구 모음.

## 목차

1. `app.help()` / `repr(app)` — Discovery
2. `app.debug` — 디버깅 도구
3. `app.lint()` — 문서 품질 체크
4. `app.template` — 문서 템플릿 save/apply
5. `app.config` — App 기본 선호도


## 1. Discovery — 어떤 기능이 있는지 모를 때

```python
from hwpapi.core import App
app = App()

# 18개 accessor 를 카테고리별로 출력
app.help()
# ════════════════════════════════════════════════════════════════════
#  hwpapi · App 사용 가능 API
# ════════════════════════════════════════════════════════════════════
#
# · Navigation & Selection
#     app.move              — 커서 이동 (.doc.top(), .line.end() ...)
#     app.sel               — 현재 선택 제어 (.current_paragraph(), ...)
# · Collections
#     app.documents         — 열린 문서 컬렉션
#     ...

# 상태 요약 한 줄
repr(app)
# "App(visible=True, version='13.0.0', docs=2, page=5/20)"

# Python REPL 에서 accessor 메소드 탐색 — tab 보조
# app.preset.<TAB>   → striped_rows, title_box, table_header, ...
```


## 2. app.debug — 디버깅 도구

### 2.1 state() — 현재 상태 전체 덤프

```python
s = app.debug.state()
# {
#   "cursor": {"doc_id": 1, "para": 15, "pos": 8},
#   "page": {"current": 3, "total": 10},
#   "selection": {"text": "Hello world", "length": 11},
#   "charshape_summary": {
#       "fontsize": 1100, "bold": True, "italic": False,
#       "text_color": "#000000", "shade_color": None,
#   },
#   "in_table": False,
#   "documents_open": 2,
#   "visible": True,
#   "version": "13.0.0",
#   "filepath": "report.hwp",
# }

app.debug.print()   # 위 내용을 예쁘게 출력
```

### 2.2 timing(fn) — 함수 호출 시간 측정

```python
# 단일 함수
r = app.debug.timing(app.insert_text, "테스트 텍스트")
print(r)
# {
#   'result': <...>,
#   'elapsed_ms': 12.34,
#   'exception': None,
#   'success': True,
# }

# 어느 작업이 느린가?
for fn, args in [
    (app.insert_text, ("hi",)),
    (app.find_text, ("HWP",)),
    (app.lint, ()),
]:
    r = app.debug.timing(fn, *args)
    print(f"{fn.__name__:20s} {r['elapsed_ms']:.2f}ms")
```

### 2.3 trace() — COM 호출 로그 (context manager)

```python
# 어떤 COM Run() 호출이 일어나는지 추적
with app.debug.trace():
    app.preset.title_box(text="Test")

# 출력:
# [trace   0] Run('TableCreate') → True (15.2ms)
# [trace   1] Run('TableColBegin') → True (0.3ms)
# [trace   2] Run('TableCellBlock') → True (0.2ms)
# [trace   3] Run('CellFill') → True (1.1ms)
# [trace   4] Run('Cancel') → True (0.2ms)
# ... (총 12 개 Run 호출)
# [trace] total: 12 Run() calls
```

debug 필요 없을 때:
```python
with app.debug.trace(verbose=False):
    # 조용히 카운트만
    ...
```


## 3. app.lint() — 문서 품질 체크

callable accessor. `app.lint()` 호출 시 `LintReport` 반환.

```python
report = app.lint()
print(report.summary)
# Document lint — 7 issue(s)
#   Total: 42 paragraphs, 118 sentences, 8253 chars
#   Long sentences: 3
#   Empty paragraphs: 2
#   Double spaces: 1 para(s)

report.has_issues               # True
report.issue_count              # 7

# 세부 — 각 항목
report.long_sentences           # [(para_idx, length), ...]
report.long_paragraphs          # [(para_idx, char_count), ...]
report.empty_paragraphs         # [1, 17]
report.double_spaces            # [para_idx, ...]
report.trailing_whitespace      # [para_idx, ...]

report.total_paragraphs
report.total_sentences
report.total_chars
```

### 임계값 조정

```python
# 기본: 문장 80자 초과, 문단 500자 초과
report = app.lint(
    long_sentence_threshold=60,    # 더 엄격
    long_paragraph_threshold=300,
)
```

### CI / 대량 품질 감사

```python
import csv
from pathlib import Path
from hwpapi.core import App

def audit_directory(root: str, out_csv: str):
    app = App(is_visible=False)
    rows = []
    with app.batch_mode():
        for path in Path(root).rglob("*.hwp"):
            try:
                app.open(str(path))
                r = app.lint()
                rows.append({
                    "path": str(path.relative_to(root)),
                    "issues": r.issue_count,
                    "sentences": r.total_sentences,
                    "long_sentences": len(r.long_sentences),
                    "long_paragraphs": len(r.long_paragraphs),
                    "empty_paragraphs": len(r.empty_paragraphs),
                })
                app.close()
            except Exception as e:
                print(f"skip {path}: {e}")

    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader()
        w.writerows(rows)

    # Top offenders
    for r in sorted(rows, key=lambda x: -x["issues"])[:10]:
        print(f"{r['issues']:3d} issues — {r['path']}")
```


## 4. app.template — 문서 템플릿 save/apply

현재 문서의 주요 **서식 설정** 을 JSON 으로 추출해서 다른 문서에 재적용.

### 4.1 template.save(path)

```python
# 예쁘게 세팅된 문서에서
app.set_charshape(facename_hangul="함초롬바탕", fontsize=1100, text_color="#000000")
app.set_parashape(line_spacing=160, indent=0)

# 현재 설정 스냅샷
data = app.template.save("company_style.json")
# {
#   "format": "hwpapi-template",
#   "version": "1.0",
#   "charshape": {"fontsize": 1100, "facename_hangul": "함초롬바탕", ...},
#   "parashape": {"line_spacing": 160, ...},
#   "page": {...},
# }
```

### 4.2 template.apply(path)

```python
# 새 문서
app2 = App()
app2.new_document()
app2.insert_text("여기는 기본 서식")

# 템플릿 적용
app2.template.apply("company_style.json")
# → charshape/parashape 가 일괄 반영

# 이제 새로 삽입되는 텍스트는 template 스타일
app2.insert_text("여기부터는 회사 스타일")
```

### 4.3 대량 문서에 일괄 적용

```python
with app.batch_mode():
    for path in Path("reports/").glob("*.hwp"):
        app.open(str(path))
        app.template.apply("company_style.json")
        app.save()
```


## 5. app.config — App 기본 선호도

각 App 인스턴스의 기본값을 저장. `~/.hwpapirc` 에 persist.

### 5.1 설정 / 읽기

```python
app.config.default_font = "함초롬바탕"
app.config.default_size = 11
app.config.default_line_spacing = 160
app.config.default_table_style = "header_sky"
app.config.palette = {
    "primary": "#003366",
    "accent": "#FF6600",
    "bg_alt": "#EEEEEE",
    "text_muted": "#666666",
}

# 일괄 업데이트
app.config.update(
    default_font="함초롬바탕",
    default_size=11,
)

# 값 읽기
app.config.default_font            # "함초롬바탕"
app.config.palette["primary"]      # "#003366"

# 전체 dict
app.config.to_dict()
```

### 5.2 디스크 저장/로드

```python
# 홈 디렉터리에 저장
app.config.save("~/.hwpapirc")

# 다른 세션에서
from hwpapi.core import App
app2 = App()
app2.config.load("~/.hwpapirc")
# 이제 app2.config.default_font 등이 복원됨
```

### 5.3 팀 공유 config

```python
# 회사 표준을 .json 으로 버전 관리
app.config.save("./company_hwpapirc.json")

# 동료는
app.config.load("./company_hwpapirc.json")
app.config.default_font            # "회사표준폰트"

# reset
app.config.reset()                  # defaults 로 복원
```


## 6. 실전 조합 — 디버그 세션

```python
from hwpapi.core import App

app = App(is_visible=True)
app.open("problem.hwp")

# 1. 상태 파악
app.debug.print()

# 2. 문서 품질 체크
report = app.lint()
if report.has_issues:
    print(report.summary)
    # 긴 문장 top 3
    for para_idx, length in sorted(report.long_sentences, key=lambda x: -x[1])[:3]:
        print(f"  para {para_idx}: {length}자")
        app.move.para.start()
        for _ in range(para_idx):
            app.move.para.next()
        app.sel.current_paragraph()
        print(f"    \"{app.sel.text[:60]}...\"")
        app.sel.clear()

# 3. 어느 작업이 느린가
slow_ops = []
for fn_name in ["current_page", "text", "lint"]:
    fn = getattr(app, fn_name)
    r = app.debug.timing(lambda: fn() if callable(fn) else fn)
    slow_ops.append((fn_name, r["elapsed_ms"]))
for name, ms in sorted(slow_ops, key=lambda x: -x[1]):
    print(f"{name:20s} {ms:.1f}ms")

# 4. 특정 작업 trace
with app.debug.trace():
    app.preset.striped_rows()
# 어떤 COM 호출들이 일어나는지 정확히 파악
```
